In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("merge_keys", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type")
merge_keys = dbutils.widgets.get("merge_keys")

print("=" * 80)
print("[STEP 1] Job parameters")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  load_type         : '{load_type}'")
print(f"  merge_keys        : '{merge_keys}'")
for _name, _val in {"landing_timestamp": landing_timestamp, "table_name": table_name}.items():
    if not _val.strip():
        print(f"  WARNING: required parameter '{_name}' is empty")
print("=" * 80)

In [0]:
source_table = f"databeli_adb.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("[STEP 2] Reading bronze data")
print(f"  Source table : {source_table}")
print("  Equivalent query executed:")
print(f"    SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Rows read : {bronze_count}")
print(f"  Columns   : {len(bronze_df.columns)} -> {bronze_df.columns}")
if bronze_count == 0:
    print(f"  WARNING: no rows found for landing_timestamp '{landing_timestamp}'")
print("=" * 80)

In [0]:
target_table = f"databeli_adb.silver.{table_name}"
normalized_load_type = (load_type or 'FULL').strip().upper()
merge_keys_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print("=" * 80)
print("[STEP 3] Validation and transformation")
print(f"  Target table         : {target_table}")
print(f"  Raw load_type        : '{load_type}' -> normalized: '{normalized_load_type}'")
print(f"  Merge keys (parsed)  : {merge_keys_list}")

if normalized_load_type == 'MERGE':
    if not merge_keys_list:
        raise ValueError("merge keys should be provided for the 'MERGE' load type")

    missing_merge_keys = [key for key in merge_keys_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge keys are not found in source date: {missing_merge_keys}")
    print("  Merge keys validated: all keys present in source columns")
    
from pyspark.sql.functions import *
updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp())\
                    .withColumn("updated_timestamp", current_timestamp())

print("  Equivalent query executed:")
print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS updated_timestamp")
print(f"    FROM {source_table} WHERE {filter_condition}")
print(f"  Columns after transform : {len(updated_df.columns)} -> {updated_df.columns}")
print("  Sample rows (5):")
updated_df.show(5, truncate=False)
print("=" * 80)


In [0]:
print("=" * 80)
print("[STEP 4] Writing to silver")
print(f"  Target table : {target_table}")
print(f"  Load type    : {normalized_load_type}")

table_exists = spark.catalog.tableExists(target_table)
rows_before = spark.table(target_table).count() if table_exists else 0
print(f"  Table exists before write : {table_exists}")
print(f"  Rows before write         : {rows_before}")

select_sql = f"SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS updated_timestamp FROM {source_table} WHERE {filter_condition}"

if normalized_load_type == 'FULL':
    print("  Equivalent query executed:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS {select_sql}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif normalized_load_type == 'APPEND':
    print("  Equivalent query executed:")
    print(f"    INSERT INTO {target_table} {select_sql}")
    updated_df.write.mode("append").saveAsTable(target_table)

elif normalized_load_type == 'MERGE':
    if not table_exists:
        print("  Target table does not exist -> creating it with the full source data")
        print("  Equivalent query executed:")
        print(f"    CREATE OR REPLACE TABLE {target_table} AS {select_sql}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        source_view = 'silver_merge_source'
        updated_df.createOrReplaceTempView(source_view)

        merge_condition = " AND ".join(
                                    [f"target.`{key}` = source.`{key}`" for key in merge_keys_list]
                                )
        update_assignments = ", ".join(
                                [
                                    f"target.`{column}` = source.`{column}`"
                                    for column in updated_df.columns
                                    if column != "insert_timestamp"
                                ]
                                )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
                  MERGE INTO {target_table} AS target
                  USING {source_view} AS source
                  ON {merge_condition}
                  WHEN MATCHED THEN UPDATE SET {update_assignments}
                  WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
                  """
        print(f"  Temp view '{source_view}' created from: {select_sql}")
        print(f"  Merge condition : {merge_condition}")
        print("  Query executed:")
        print(merge_sql)

        merge_result = spark.sql(merge_sql)
        print("  MERGE metrics:")
        merge_result.show(truncate=False)
else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

rows_after = spark.table(target_table).count()
print(f"  Rows after write  : {rows_after}")
print(f"  Net row change    : {rows_after - rows_before}")
print("=" * 80)


In [0]:
record_count = updated_df.count()
print(f"[DONE] Bronze to silver load completed. Records processed: {record_count}")
dbutils.notebook.exit(record_count)